# Customer Churn Prediction – Telco Dataset

**Task:** This notebook was written by a junior analyst. It runs without errors, but it contains **several mistakes** in data handling, modelling, evaluation and interpretation.

Your job:
1. Read and run the notebook.
2. Find every mistake you can. For each one, write: **the cell**, **what is wrong**, **why it matters**, and **how to fix it**.
3. Produce a corrected version of the notebook and compare the results.

Submit your findings as a Word document plus the corrected notebook.

## 1. Load the data

In [3]:
import pandas as pd
#import numpy as np
#mport matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
#from sklearn.metrics import accuracy_score
from sklearn.metrics import classification_report

url = "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv"
df = pd.read_csv(url)
df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   object 
 1   gender            7043 non-null   object 
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   object 
 4   Dependents        7043 non-null   object 
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   object 
 7   MultipleLines     7043 non-null   object 
 8   InternetService   7043 non-null   object 
 9   OnlineSecurity    7043 non-null   object 
 10  OnlineBackup      7043 non-null   object 
 11  DeviceProtection  7043 non-null   object 
 12  TechSupport       7043 non-null   object 
 13  StreamingTV       7043 non-null   object 
 14  StreamingMovies   7043 non-null   object 
 15  Contract          7043 non-null   object 
 16  PaperlessBilling  7043 non-null   object 


## 2. Cleaning
`TotalCharges` is stored as text, so we convert it to a number.

In [9]:
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
#df['TotalCharges'] = df['TotalCharges'].fillna(df['TotalCharges'].mean())
df['TotalCharges'] = df['TotalCharges'].fillna(df['TotalCharges'].median()) # median better with the out layers
df = df.drop(columns=['customerID'])
df.isnull().sum().sum()

np.int64(0)

# 2.1 replacing the labelencoder with somthing beter


In [10]:
df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0})
df = pd.get_dummies(df, drop_first=True) # transfare all the categore colmn for true false

## 3. Exploratory analysis

In [16]:
#churn_by_contract = df.groupby('Contract')['Churn'].apply(lambda s: (s == 'Yes').mean())
#churn_by_contract.plot(kind='bar', title='Churn rate by contract type')
#plt.ylabel('Churn rate')
#plt.show()
#churn_by_contract
df.info()
df.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 31 columns):
 #   Column                                 Non-Null Count  Dtype  
---  ------                                 --------------  -----  
 0   SeniorCitizen                          7043 non-null   int64  
 1   tenure                                 7043 non-null   int64  
 2   MonthlyCharges                         7043 non-null   float64
 3   TotalCharges                           7043 non-null   float64
 4   Churn                                  7043 non-null   int64  
 5   gender_Male                            7043 non-null   bool   
 6   Partner_Yes                            7043 non-null   bool   
 7   Dependents_Yes                         7043 non-null   bool   
 8   PhoneService_Yes                       7043 non-null   bool   
 9   MultipleLines_No phone service         7043 non-null   bool   
 10  MultipleLines_Yes                      7043 non-null   bool   
 11  Inte

,SeniorCitizen,tenure,MonthlyCharges,TotalCharges,Churn,gender_Male,Partner_Yes,Dependents_Yes,PhoneService_Yes,MultipleLines_No phone service,...,StreamingTV_No internet service,StreamingTV_Yes,StreamingMovies_No internet service,StreamingMovies_Yes,Contract_One year,Contract_Two year,PaperlessBilling_Yes,PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check
0,0,1,29.85,29.85,0,False,True,False,False,True,...,False,False,False,False,False,False,True,False,True,False
1,0,34,56.95,1889.50,0,True,False,False,True,False,...,False,False,False,False,True,False,False,False,False,True
2,0,2,53.85,108.15,1,True,False,False,True,False,...,False,False,False,False,False,False,True,False,False,True
3,0,45,42.30,1840.75,0,True,False,False,False,True,...,False,False,False,False,True,False,False,False,False,False
4,0,2,70.70,151.65,1,False,False,False,True,False,...,False,False,False,False,False,False,True,False,True,False


**From** the chart, **month-to-month customers have the lowest churn rate**, so contract length does not seem to be a strong driver of churn.

In [17]:
#df['Churn'].value_counts()

# we don`t need it sense transfare all the categore colmn for true false

,count
Churn,
0,5174
1,1869


## 4. Feature engineering

In [ ]:
# Churn rate per contract type as a numeric feature
#df['contract_churn_rate'] = df['Contract'].map(churn_by_contract)

In [ ]:
#encoder = LabelEncoder()
#for col in df.select_dtypes(include='object').columns:
#    df[col] = encoder.fit_transform(df[col])

#df.head()

## 5. Prepare data for modelling

In [20]:
X = df.drop(columns=['Churn'])
y = df['Churn']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y ) # we should split the data first

scaler = StandardScaler()
#X_scaled = scaler.fit_transform(X)
                                          #it should be like this
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)
# X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2) this step should be befor

# this part of code we will full change it because It performs prediction using X_train and calculates the loss against y_train. and we chang all that with the data lacked using GridSearchCV

In [23]:
# new code
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train) # Apply `fit_transform` to the training data (X_train) so that the model learns from it and applies the transformation.
X_test_scaled = scaler.transform(X_test)  # Apply `transform` to the testing data (X_test) To perform the adjustment based on the characteristics of the training data.
# make the data ready for trainig

## 6. Train the model

In [24]:
#new code :
param_grid = {'C': [0.001, 0.01, 0.1, 1, 10, 100]}
grid = GridSearchCV(LogisticRegression(max_iter=1000), param_grid, cv=5, scoring='f1')
grid.fit(X_train_scaled, y_train)

GridSearchCV(cv=5, estimator=LogisticRegression(max_iter=1000),
             param_grid={'C': [0.001, 0.01, 0.1, 1, 10, 100]}, scoring='f1')

In [31]:
print(f"the best model of c is C{grid.best_params_['C']}")

the best model of c is C1


In [28]:
# new code
best_model = grid.best_estimator_
preds = best_model.predict(X_test_scaled)
print("\nFinal performance for the new model after fixing : ")
print(classification_report(y_test, preds))


Final performance for the new model after fixing : 
              precision    recall  f1-score   support

           0       0.85      0.92      0.89      1034
           1       0.73      0.56      0.63       375

    accuracy                           0.83      1409
   macro avg       0.79      0.74      0.76      1409
weighted avg       0.82      0.83      0.82      1409



In [ ]:
#model = LogisticRegression(max_iter=1000)
#model.fit(X_train, y_train)

## 7. Test performance

In [ ]:
#preds = model.predict(X_train)
#print("Test accuracy:", accuracy_score(y_train, preds))

## 8. Hyperparameter tuning
We try several values of `C` and keep the best one.

In [ ]:
#best_c, best_acc = None, 0
#for c in [0.001, 0.01, 0.1, 1, 10, 100]:
   # m = LogisticRegression(C=c, max_iter=1000).fit(X_train, y_train)
  #  acc = accuracy_score(y_test, m.predict(X_test))
  #  if acc > best_acc:
   #     best_c, best_acc = c, acc

#print("Best C:", best_c, "| Final test accuracy:", best_acc)

## 9. Conclusion

- The model reaches around **80% accuracy**, which is excellent and means it is ready for production.
- Accuracy is the right metric here because the classes are balanced.
- Contract type is not an important factor for churn.
- The final test accuracy above is an unbiased estimate of how the model will perform on new customers.